# Multi-Qubit Gates & Entanglement

Create Bell states, explore CNOT, SWAP, and Toffoli gates.

**Objectives:**
- Build entangled states using CNOT and verify correlations
- Create all four Bell states
- Use SWAP and Toffoli (CCNOT) gates
- Understand how entanglement differs from classical correlation

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->


In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import matplotlib.pyplot as plt

from lib.circuits import bell_pair, ghz_state
from lib.utils.results import parse_counts

device = LocalSimulator()

## 1. CNOT Gate and the Bell Pair

CNOT (Controlled-NOT) flips the target qubit if and only if the control qubit is |1>.
Combined with Hadamard, it creates entanglement.

In [ ]:
# CNOT truth table: verify on all 4 basis states
print("CNOT Truth Table (control=q0, target=q1):")
print(f"{'Input':<10} {'Output':<10}")
print("-" * 20)

for input_state in ['00', '01', '10', '11']:
    circuit = Circuit()
    if input_state[0] == '1':
        circuit.x(0)
    if input_state[1] == '1':
        circuit.x(1)
    circuit.cnot(0, 1)
    
    result = device.run(circuit, shots=10).result()
    output = list(result.measurement_counts.keys())[0]
    print(f"|{input_state}>    ->  |{output}>")

In [ ]:
# Bell pair: H then CNOT creates (|00> + |11>) / sqrt(2)
circuit = bell_pair()
print(circuit)

result = device.run(circuit, shots=2000).result()
counts = parse_counts(result)

print(f"\nMeasurement counts: {dict(counts)}")
print("\nVerifying entanglement: every measurement shows both qubits agree.")

for bitstring in counts:
    assert bitstring[0] == bitstring[1], f"Correlation broken: {bitstring}"
print("All measurements show perfect qubit correlation.")

## 2. All Four Bell States

There are four maximally entangled two-qubit states (Bell basis):
- |Phi+> = (|00> + |11>) / sqrt(2) -- H, CNOT
- |Phi-> = (|00> - |11>) / sqrt(2) -- Z, H, CNOT
- |Psi+> = (|01> + |10>) / sqrt(2) -- X on q1, H, CNOT
- |Psi-> = (|01> - |10>) / sqrt(2) -- X on q1, Z, H, CNOT

In [ ]:
# Create all four Bell states and compare their measurement statistics
bell_states = {
    "|Phi+>": Circuit().h(0).cnot(0, 1),
    "|Phi->": Circuit().h(0).z(0).cnot(0, 1),
    "|Psi+>": Circuit().h(0).cnot(0, 1).x(1),
    "|Psi->": Circuit().h(0).z(0).cnot(0, 1).x(1),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, (name, circuit) in zip(axes, bell_states.items()):
    result = device.run(circuit, shots=2000).result()
    counts = result.measurement_counts
    
    labels = sorted(counts.keys())
    values = [counts[label] / 2000 for label in labels]
    
    ax.bar(labels, values, color='#232f3e')
    ax.set_title(name)
    ax.set_ylim(0, 0.7)
    ax.set_ylabel('Probability')

plt.suptitle('The Four Bell States', fontsize=12)
plt.tight_layout()
plt.show()

print("|Phi+/-> produce |00> and |11> (qubits agree)")
print("|Psi+/-> produce |01> and |10> (qubits disagree)")

## 3. GHZ State (Multi-Qubit Entanglement)

The GHZ state generalizes the Bell pair to n qubits: (|00...0> + |11...1>) / sqrt(2).
Measuring any single qubit collapses all the others.

In [ ]:
# Create and verify GHZ states of increasing size
for n in [3, 4, 5]:
    circuit = ghz_state(n_qubits=n)
    result = device.run(circuit, shots=1000).result()
    counts = result.measurement_counts
    
    all_zeros = '0' * n
    all_ones = '1' * n
    
    print(f"{n}-qubit GHZ: {dict(counts)}")
    assert set(counts.keys()) == {all_zeros, all_ones}, f"Unexpected states in {n}-qubit GHZ!"

print("\nAll GHZ states produce only all-zeros or all-ones outcomes.")

## 4. SWAP Gate

The SWAP gate exchanges the states of two qubits. It can be decomposed into three CNOTs.

In [ ]:
# Prepare q0=|1>, q1=|0>, then SWAP
circuit = Circuit().x(0).swap(0, 1)
print(circuit)

result = device.run(circuit, shots=100).result()
counts = result.measurement_counts
print(f"\nAfter SWAP: {dict(counts)}")
print("q0 was |1>, q1 was |0> -> after SWAP: q0=|0>, q1=|1> -> outcome '01'")

# Verify: SWAP is equivalent to 3 CNOTs
circuit_3cnot = Circuit().x(0).cnot(0, 1).cnot(1, 0).cnot(0, 1)
result2 = device.run(circuit_3cnot, shots=100).result()
counts2 = result2.measurement_counts
print(f"3-CNOT decomposition: {dict(counts2)}")
assert counts == counts2, "SWAP and 3-CNOT should give identical results"
print("Confirmed: SWAP = CNOT * CNOT * CNOT")

## 5. Toffoli Gate (CCNOT)

The Toffoli gate has two controls and one target. It flips the target only when both controls are |1>.
This gate is universal for classical reversible computation.

In [ ]:
# Toffoli truth table
print("Toffoli Truth Table (controls=q0,q1, target=q2):")
print(f"{'Input':<10} {'Output':<10} {'Target flipped?'}")
print("-" * 35)

for q0 in [0, 1]:
    for q1 in [0, 1]:
        for q2 in [0, 1]:
            circuit = Circuit()
            if q0:
                circuit.x(0)
            if q1:
                circuit.x(1)
            if q2:
                circuit.x(2)
            circuit.ccnot(0, 1, 2)
            
            result = device.run(circuit, shots=10).result()
            output = list(result.measurement_counts.keys())[0]
            flipped = 'YES' if output[2] != str(q2) else 'no'
            print(f"|{q0}{q1}{q2}>    ->  |{output}>     {flipped}")

## 6. Exercises

In [ ]:
# Exercise 1: Create the |Phi-> Bell state (|00> - |11>) / sqrt(2)
# using H on qubit 0, Z on qubit 0, then CNOT.
# Verify it produces only |00> and |11> outcomes.

# TODO: Your code here
# phi_minus = Circuit()...
# result = device.run(phi_minus, shots=1000).result()
# assert set(result.measurement_counts.keys()) == {'00', '11'}

In [ ]:
# Exercise 2: Build a quantum AND gate using Toffoli.
# The AND of two input qubits should appear on a third (target) qubit.
# Initialize target to |0>, apply Toffoli, measure target.
# Test all 4 input combinations and verify AND logic.

# TODO: Your code here
# for a, b in [(0,0), (0,1), (1,0), (1,1)]:
#     circuit = Circuit()
#     if a: circuit.x(0)
#     if b: circuit.x(1)
#     circuit.ccnot(0, 1, 2)
#     ...

## Summary

- **CNOT**: conditional flip, creates entanglement when control is in superposition
- **Bell states**: 4 maximally entangled 2-qubit states, foundation of quantum protocols
- **GHZ state**: n-qubit entanglement where all qubits are correlated
- **SWAP**: exchanges qubit states, decomposes into 3 CNOTs
- **Toffoli (CCNOT)**: 2-control gate, implements reversible classical logic
- CNOT + single-qubit gates form a universal gate set

**Next:** [04-measurement-statistics.ipynb](04-measurement-statistics.ipynb) -- understand shot-based measurement and statistical accuracy.